# Документ OCR pipeline (Google Colab)

Рабочий baseline для тестового задания ML Engineer:
1. Загружает фото документа
2. Выравнивает документ (perspective transform)
3. Распознаёт текст (EasyOCR, локально, без токенов)
4. Извлекает структурированные поля (ФИО, дата рождения, номер документа)
5. Сохраняет выровненное изображение, изображение с боксами и JSON


In [ ]:
!pip -q install opencv-python-headless easyocr matplotlib rapidfuzz

In [ ]:
import re
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import easyocr
import torch
from google.colab import files

In [ ]:
# --- Геометрия: выравнивание документа ---
def order_points(pts):
    rect = np.zeros((4, 2), dtype='float32')
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect


def four_point_transform(image, pts):
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = int(max(heightA, heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]
    ], dtype='float32')

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    return warped


def align_document(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(gray, 60, 180)

    contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:10]

    doc_cnt = None
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:
            doc_cnt = approx.reshape(4, 2)
            break

    if doc_cnt is None:
        # fallback: если контур не найден, возвращаем исходное изображение
        return image_bgr

    aligned = four_point_transform(image_bgr, doc_cnt.astype('float32'))
    return aligned

In [ ]:
# --- OCR + визуализация ---
reader = easyocr.Reader(['ru', 'en'], gpu=torch.cuda.is_available())


def run_ocr(image_bgr):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    results = reader.readtext(rgb, detail=1)
    return results


def draw_boxes(image_bgr, ocr_results):
    out = image_bgr.copy()
    for item in ocr_results:
        box, text, conf = item
        pts = np.array(box, dtype=np.int32)
        cv2.polylines(out, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
        x, y = pts[0]
        cv2.putText(out, f'{text[:25]} ({conf:.2f})', (x, max(0, y - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
    return out

In [ ]:
# --- Извлечение полей из OCR текста ---
date_pattern = re.compile(r'\b(\d{2}[./-]\d{2}[./-]\d{4})\b')
doc_number_pattern = re.compile(r'\b(\d{2}\s?\d{2}\s?\d{6}|\d{9,12})\b')
fio_pattern = re.compile(r'^[А-ЯЁ][А-ЯЁ-]+(?:\s+[А-ЯЁ][А-ЯЁ-]+){1,2}$')


def clean_text(s):
    return re.sub(r'\s+', ' ', s.strip())


def extract_fields(ocr_results):
    lines = [clean_text(x[1]) for x in ocr_results if x[2] > 0.25]

    joined = ' '.join(lines)

    birth_date = None
    date_match = date_pattern.search(joined)
    if date_match:
        birth_date = date_match.group(1).replace('-', '.').replace('/', '.')

    doc_number = None
    num_match = doc_number_pattern.search(joined.replace('O', '0').replace('o', '0'))
    if num_match:
        doc_number = num_match.group(1)

    fio = None
    for line in lines:
        c = line.upper()
        if fio_pattern.match(c):
            fio = c
            break

    return {
        'full_name': fio,
        'birth_date': birth_date,
        'document_number': doc_number
    }

In [ ]:
# --- Основной пайплайн ---
def process_document_image(image_path, out_dir='outputs'):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True, parents=True)

    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise ValueError(f'Не удалось прочитать изображение: {image_path}')

    aligned = align_document(image_bgr)
    ocr_results = run_ocr(aligned)
    annotated = draw_boxes(aligned, ocr_results)
    fields = extract_fields(ocr_results)

    stem = Path(image_path).stem
    aligned_path = out_dir / f'{stem}_aligned.jpg'
    annotated_path = out_dir / f'{stem}_annotated.jpg'
    json_path = out_dir / f'{stem}_fields.json'

    cv2.imwrite(str(aligned_path), aligned)
    cv2.imwrite(str(annotated_path), annotated)

    payload = {
        'input_image': str(image_path),
        'aligned_image': str(aligned_path),
        'annotated_image': str(annotated_path),
        'fields': fields,
        'ocr': [
            {
                'box': item[0],
                'text': item[1],
                'confidence': float(item[2])
            } for item in ocr_results
        ]
    }

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return payload

In [ ]:
# --- Загрузка файла в Colab и запуск ---
uploaded = files.upload()
image_name = next(iter(uploaded.keys()))

result = process_document_image(image_name, out_dir='outputs')
print(json.dumps(result['fields'], ensure_ascii=False, indent=2))

aligned_rgb = cv2.cvtColor(cv2.imread(result['aligned_image']), cv2.COLOR_BGR2RGB)
annotated_rgb = cv2.cvtColor(cv2.imread(result['annotated_image']), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.title('Aligned')
plt.imshow(aligned_rgb)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('Detections + OCR')
plt.imshow(annotated_rgb)
plt.axis('off')
plt.show()

In [ ]:
# --- Скачать артефакты на локальную машину ---
files.download(result['aligned_image'])
files.download(result['annotated_image'])
files.download(f"outputs/{Path(image_name).stem}_fields.json")